## Audit new drift-aware backtest engine

### Setup

In [1]:
import numpy as np
import pandas as pd

from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet
from alpha_research.portfolio import build_factor_target_weights


panel = load_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}

config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

### Run both engines

In [2]:
audit_results = {}

return_panel = panel[
    ["date", "ticker", "forward_ret_1d"]
].copy()

for factor_name, factor_column in factor_columns.items():
    target_weights = build_factor_target_weights(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    legacy_daily, legacy_holdings = run_long_short_backtest(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    drift_daily, drift_holdings = run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=target_weights,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )

    audit_results[factor_name] = {
        "targets": target_weights,
        "legacy_daily": legacy_daily,
        "legacy_holdings": legacy_holdings,
        "drift_daily": drift_daily,
        "drift_holdings": drift_holdings,
    }

### Verify the results

Both engines should match the target weights exactly on rebalance dates.

In [3]:
def maximum_weight_difference(
    left: pd.DataFrame,
    right: pd.DataFrame,
) -> float:
    comparison = left[["date", "ticker", "weight"]].merge(
        right[["date", "ticker", "weight"]],
        on=["date", "ticker"],
        how="outer",
        suffixes=("_left", "_right"),
    )

    comparison[["weight_left", "weight_right"]] = comparison[
        ["weight_left", "weight_right"]
    ].fillna(0.0)

    return float((comparison["weight_left"] - comparison["weight_right"]).abs().max())


audit_check_rows = []

for factor_name, result in audit_results.items():
    targets = result["targets"]
    target_dates = targets["date"].unique()

    legacy_rebalance_holdings = result["legacy_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    drift_rebalance_holdings = result["drift_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    daily_comparison = result["legacy_daily"][
        ["date", "is_rebalance", "gross_return"]
    ].merge(
        result["drift_daily"][["date", "is_rebalance", "gross_return"]],
        on="date",
        how="inner",
        suffixes=("_legacy", "_drift"),
        validate="one_to_one",
    )

    gross_difference = (
        daily_comparison["gross_return_drift"] - daily_comparison["gross_return_legacy"]
    ).abs()

    rebalance_mask = daily_comparison["is_rebalance_legacy"]

    audit_check_rows.append(
        {
            "factor": factor_name,
            "same_daily_dates": (
                len(daily_comparison)
                == len(result["legacy_daily"])
                == len(result["drift_daily"])
            ),
            "rebalance_dates": len(target_dates),
            "max_legacy_target_mismatch": (
                maximum_weight_difference(
                    legacy_rebalance_holdings,
                    targets,
                )
            ),
            "max_drift_target_mismatch": (
                maximum_weight_difference(
                    drift_rebalance_holdings,
                    targets,
                )
            ),
            "max_rebalance_gross_return_difference": (
                gross_difference.loc[rebalance_mask].max()
            ),
            "mean_non_rebalance_gross_return_difference": (
                gross_difference.loc[~rebalance_mask].mean()
            ),
        }
    )

audit_checks = pd.DataFrame(audit_check_rows).set_index("factor")

audit_checks

,same_daily_dates,rebalance_dates,max_legacy_target_mismatch,max_drift_target_mismatch,max_rebalance_gross_return_difference,mean_non_rebalance_gross_return_difference
factor,,,,,,
12-1 Momentum,True,578,0.0,0.0,0.0,0.000349
Realised Volatility,True,578,0.0,0.0,0.0,0.000262


#### Compare economic results

In [4]:
summary_rows = []

for factor_name, result in audit_results.items():
    for engine_name, daily in {
        "Legacy": result["legacy_daily"],
        "Drift-aware": result["drift_daily"],
    }.items():
        gross = summarise_backtest(
            daily,
            return_column="gross_return",
        ).iloc[0]

        net = summarise_backtest(
            daily,
            return_column="net_return",
        ).iloc[0]

        summary_rows.append(
            {
                "factor": factor_name,
                "engine": engine_name,
                "gross_total_return": gross["total_return"],
                "net_total_return": net["total_return"],
                "gross_annualised_return": (gross["annualised_return"]),
                "net_annualised_return": (net["annualised_return"]),
                "net_annualised_volatility": (net["annualised_volatility"]),
                "gross_sharpe": gross["sharpe_ratio"],
                "net_sharpe": net["sharpe_ratio"],
                "average_rebalance_turnover": (net["average_rebalance_turnover"]),
                "total_transaction_cost": (net["total_transaction_cost"]),
            }
        )

audit_summary = (
    pd.DataFrame(summary_rows).sort_values(["factor", "engine"]).reset_index(drop=True)
)

audit_summary.round(4)

,factor,engine,gross_total_return,net_total_return,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,average_rebalance_turnover,total_transaction_cost
0,12-1 Momentum,Drift-aware,0.4804,0.1038,0.0348,0.0086,0.2119,0.2683,0.1475,0.5080,0.2936
1,12-1 Momentum,Legacy,0.5957,0.2379,0.0416,0.0188,0.2113,0.2995,0.1947,0.4393,0.2539
2,Realised Volatility,Drift-aware,5.5496,4.1786,0.1781,0.1542,0.2395,0.8042,0.7185,0.4063,0.2348
3,Realised Volatility,Legacy,5.5559,4.3711,0.1782,0.1579,0.2399,0.8036,0.7310,0.3448,0.1993


#### Isolate the change

In [5]:
metric_columns = [
    "gross_annualised_return",
    "net_annualised_return",
    "net_annualised_volatility",
    "gross_sharpe",
    "net_sharpe",
    "average_rebalance_turnover",
    "total_transaction_cost",
]

legacy_summary = audit_summary.loc[audit_summary["engine"] == "Legacy"].set_index(
    "factor"
)

drift_summary = audit_summary.loc[audit_summary["engine"] == "Drift-aware"].set_index(
    "factor"
)

audit_deltas = drift_summary[metric_columns] - legacy_summary[metric_columns]

audit_deltas.columns = [f"change_in_{column}" for column in audit_deltas.columns]

audit_deltas.round(4)

,change_in_gross_annualised_return,change_in_net_annualised_return,change_in_net_annualised_volatility,change_in_gross_sharpe,change_in_net_sharpe,change_in_average_rebalance_turnover,change_in_total_transaction_cost
factor,,,,,,,
12-1 Momentum,-0.0068,-0.0101,0.0006,-0.0312,-0.0472,0.0687,0.0397
Realised Volatility,-0.0001,-0.0037,-0.0004,0.0006,-0.0125,0.0615,0.0355


### Engine audit conclusion

The drift-aware engine reproduces the legacy target portfolios exactly on rebalance dates. Rebalance-day gross returns also match exactly, confirming that the factor signals, stock selection, and target-weight construction are unchanged.

Between rebalances, the drift-aware engine carries forward holdings whose weights evolve with asset returns. This produces non-zero return differences relative to the legacy constant-weight convention and increases measured rebalance turnover.

The legacy implementation understated average rebalance turnover by approximately 0.069 for momentum and 0.062 for realised volatility. After realistic drift accounting, momentum's net Sharpe declines from approximately 0.195 to 0.148, while realised volatility's net Sharpe declines from 0.731 to 0.719.

All subsequent portfolio experiments will therefore use the drift-aware target-weight engine. Legacy results will be retained only as historical benchmarks.

## First multi-factor baseline

A 50/50 combination of momentum and realised volatility, independent sleeves

In [6]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_sleeve_target_weights,
)

In [7]:
momentum_targets = audit_results["12-1 Momentum"]["targets"]

volatility_targets = audit_results["Realised Volatility"]["targets"]

combined_targets = combine_sleeve_target_weights(
    sleeve_targets={
        "Momentum": momentum_targets,
        "Realised Volatility": volatility_targets,
    },
    sleeve_allocations={
        "Momentum": 0.5,
        "Realised Volatility": 0.5,
    },
)

combined_daily, combined_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=combined_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

#### Examine natural netting

In [8]:
combined_target_exposure = (
    combined_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

combined_target_exposure["gross_exposure"] = (
    combined_target_exposure["long_exposure"]
    + combined_target_exposure["short_exposure"]
)

combined_target_exposure["net_exposure"] = (
    combined_target_exposure["long_exposure"]
    - combined_target_exposure["short_exposure"]
)

combined_target_exposure[
    [
        "long_exposure",
        "short_exposure",
        "gross_exposure",
        "net_exposure",
        "active_positions",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,long_exposure,short_exposure,gross_exposure,net_exposure,active_positions
mean,0.7352,0.7352,1.4704,0.0,49.9204
std,0.1905,0.1905,0.3810,0.0,10.8236
min,0.0000,0.0000,0.0000,0.0,0.0000
max,1.0000,1.0000,2.0000,0.0,66.0000


#### Compare the three portfolios

In [9]:
baseline_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
}

baseline_rows = []

for portfolio_name, daily in baseline_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    baseline_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

baseline_summary = pd.DataFrame(baseline_rows).set_index("portfolio")

baseline_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712


In [10]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in baseline_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves
Momentum,1.0000,0.0225,0.6696
Realised Volatility,0.0225,1.0000,0.7572
50/50 Independent Sleeves,0.6696,0.7572,1.0000


### Experiment 1: 50/50 independent factor sleeves

The independent-sleeve portfolio allocates 50% of notional capital to the momentum portfolio and 50% to the realised-volatility portfolio, then nets their stock-level target weights.

The standalone factor returns have very low correlation (approximately 0.02), providing meaningful diversification. Factor disagreement reduces average gross exposure to approximately 1.47, while dollar neutrality is preserved.

The combined portfolio produces a net annualised return of 9.82%, volatility of 16.16%, and a net Sharpe ratio of 0.661. Its maximum drawdown of -25.66% is substantially smaller than the drawdowns of either standalone factor.

The portfolio does not exceed the realised-volatility factor's standalone Sharpe ratio, but it delivers a materially smoother risk profile through diversification and natural position netting.

## Composite score portfolio

Composite score = average of the two factor scores

In [11]:
from alpha_research.portfolio import combine_factor_scores

#### Construct composite score

In [12]:
composite_panel = panel.copy()

composite_panel["mom_vol_composite_z"] = combine_factor_scores(
    panel=composite_panel,
    factor_weights={
        "mom_12_1m_z": 0.5,
        "realised_vol_63_z": 0.5,
    },
)

composite_targets = build_factor_target_weights(
    panel=composite_panel,
    factor_column="mom_vol_composite_z",
    return_column="forward_ret_1d",
    config=config,
)

composite_daily, composite_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=composite_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

#### Examine target exposure

In [13]:
composite_target_exposure = (
    composite_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

composite_target_exposure["gross_exposure"] = (
    composite_target_exposure["long_exposure"]
    + composite_target_exposure["short_exposure"]
)

composite_target_exposure["net_exposure"] = (
    composite_target_exposure["long_exposure"]
    - composite_target_exposure["short_exposure"]
)

composite_target_exposure.agg(
    ["mean", "std", "min", "max"]
).round(4)

,long_exposure,short_exposure,active_positions,gross_exposure,net_exposure
mean,0.9118,0.9118,36.4706,1.8235,0.0
std,0.2839,0.2839,11.3553,0.5678,0.0
min,0.0000,0.0000,0.0000,0.0000,0.0
max,1.0000,1.0000,40.0000,2.0000,0.0


#### Compare portfolios

In [14]:
comparison_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
    "50/50 Composite Score": composite_daily,
}

comparison_rows = []

for portfolio_name, daily in comparison_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    comparison_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

comparison_summary = pd.DataFrame(comparison_rows).set_index("portfolio")

comparison_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
50/50 Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [15]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in comparison_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves,50/50 Composite Score
Momentum,1.0000,0.0225,0.6696,0.5510
Realised Volatility,0.0225,1.0000,0.7572,0.7886
50/50 Independent Sleeves,0.6696,0.7572,1.0000,0.9457
50/50 Composite Score,0.5510,0.7886,0.9457,1.0000


### Experiment 2: 50/50 composite factor score

The composite-score portfolio averages the standardised momentum and realised-volatility scores before ranking stocks and constructing a single long-short portfolio.

Unlike the independent-sleeve method, factor disagreement changes stock rankings rather than directly cancelling positions. The portfolio therefore maintains a higher average gross exposure of approximately 1.82.

The composite produces a net annualised return of 12.45%, volatility of 20.37%, and a net Sharpe ratio of 0.679. It modestly exceeds the independent sleeve portfolio's Sharpe ratio, but has higher turnover, transaction costs, and maximum drawdown.

The two combination methods have a return correlation of approximately 0.95, showing that they primarily express the same underlying factor information through different portfolio-construction rules.

## Controlled exposure experiment

If both combination methods carry exactly the same target gross exposure on every rebalance date, does the composite score still perform better?

In [16]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_factor_scores,
    combine_sleeve_target_weights,
    rescale_target_weights_to_gross,
)

In [17]:
composite_target_gross_schedule = (
    composite_targets
    .groupby("date")["weight"]
    .agg(lambda weights: weights.abs().sum())
    .rename("composite_target_gross")
)

controlled_sleeve_targets = rescale_target_weights_to_gross(
    target_weights=combined_targets,
    target_gross=composite_target_gross_schedule,
)

controlled_sleeve_daily, controlled_sleeve_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=controlled_sleeve_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )
)

#### Verify exposure control

In [18]:
def calculate_target_gross(
    targets: pd.DataFrame,
) -> pd.Series:
    return (
        targets.groupby("date")["weight"]
        .agg(lambda weights: weights.abs().sum())
    )


target_gross_audit = pd.concat(
    {
        "Original Independent Sleeves": calculate_target_gross(
            combined_targets
        ),
        "Equal-Exposure Independent Sleeves": calculate_target_gross(
            controlled_sleeve_targets
        ),
        "Composite Score": calculate_target_gross(
            composite_targets
        ),
    },
    axis=1,
).fillna(0.0)

target_gross_audit["controlled_minus_composite"] = (
    target_gross_audit["Equal-Exposure Independent Sleeves"]
    - target_gross_audit["Composite Score"]
)

target_gross_audit.agg(
    ["mean", "std", "min", "max"]
).round(6)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score,controlled_minus_composite
mean,1.470415,1.823529,1.823529,0.0
std,0.381037,0.567765,0.567765,0.0
min,0.000000,0.000000,0.000000,-0.0
max,2.000000,2.000000,2.000000,0.0


#### Compare performance

In [19]:
controlled_comparison_daily = {
    "Original Independent Sleeves": combined_daily,
    "Equal-Exposure Independent Sleeves": controlled_sleeve_daily,
    "Composite Score": composite_daily,
}

controlled_summary_rows = []

for portfolio_name, daily in controlled_comparison_daily.items():
    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    controlled_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross_summary[
                "annualised_return"
            ],
            "net_annualised_return": net_summary[
                "annualised_return"
            ],
            "net_annualised_volatility": net_summary[
                "annualised_volatility"
            ],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary[
                "average_rebalance_turnover"
            ],
            "total_transaction_cost": net_summary[
                "total_transaction_cost"
            ],
            "average_daily_gross_exposure": daily[
                "gross_exposure"
            ].mean(),
        }
    )

controlled_summary = (
    pd.DataFrame(controlled_summary_rows)
    .set_index("portfolio")
)

controlled_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,
Original Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
Equal-Exposure Independent Sleeves,0.1553,0.1224,0.1999,0.8228,0.6779,-0.3173,0.5742,0.3319,1.8240
Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [20]:
matched_exposure_delta = (
    controlled_summary.loc["Composite Score"]
    - controlled_summary.loc[
        "Equal-Exposure Independent Sleeves"
    ]
).rename("composite_minus_equal_exposure_sleeves")

matched_exposure_delta.round(4)

gross_annualised_return        -0.0034
net_annualised_return           0.0022
net_annualised_volatility       0.0038
gross_sharpe                   -0.0262
net_sharpe                      0.0005
max_drawdown                    0.0111
average_rebalance_turnover     -0.0971
total_transaction_cost         -0.0561
average_daily_gross_exposure    0.0002
Name: composite_minus_equal_exposure_sleeves, dtype: float64

In [21]:
controlled_gross_returns = pd.concat(
    {
        portfolio_name: daily.set_index("date")["gross_return"]
        for portfolio_name, daily
        in controlled_comparison_daily.items()
    },
    axis=1,
)

controlled_gross_returns.corr().round(4)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score
Original Independent Sleeves,1.0000,0.9858,0.9457
Equal-Exposure Independent Sleeves,0.9858,1.0000,0.9501
Composite Score,0.9457,0.9501,1.0000


#### Measure the remaining construction difference

In [22]:
target_pair = (
    controlled_sleeve_targets[
        ["date", "ticker", "weight"]
    ]
    .rename(columns={"weight": "sleeve_weight"})
    .merge(
        composite_targets[
            ["date", "ticker", "weight"]
        ].rename(
            columns={"weight": "composite_weight"}
        ),
        on=["date", "ticker"],
        how="outer",
    )
    .fillna(
        {
            "sleeve_weight": 0.0,
            "composite_weight": 0.0,
        }
    )
)

target_structure_rows = []

for date, date_targets in target_pair.groupby("date"):
    sleeve_weight = date_targets["sleeve_weight"]
    composite_weight = date_targets["composite_weight"]

    target_gross = composite_weight.abs().sum()

    long_overlap = np.minimum(
        sleeve_weight.clip(lower=0.0),
        composite_weight.clip(lower=0.0),
    ).sum()

    short_overlap = np.minimum(
        (-sleeve_weight.clip(upper=0.0)),
        (-composite_weight.clip(upper=0.0)),
    ).sum()

    same_side_overlap = long_overlap + short_overlap

    target_structure_rows.append(
        {
            "date": date,
            "target_gross": target_gross,
            "l1_weight_difference": (
                sleeve_weight - composite_weight
            ).abs().sum(),
            "same_side_weight_overlap": same_side_overlap,
            "same_side_overlap_fraction": (
                same_side_overlap / target_gross
                if target_gross > 0
                else np.nan
            ),
        }
    )

target_structure = pd.DataFrame(target_structure_rows)

target_structure[
    [
        "l1_weight_difference",
        "same_side_weight_overlap",
        "same_side_overlap_fraction",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,l1_weight_difference,same_side_weight_overlap,same_side_overlap_fraction
mean,1.2697,1.1887,0.6518
std,0.4579,0.3877,0.0605
min,0.0000,0.0000,0.3750
max,2.5000,1.6500,0.8250


### Experiment 3: Controlled gross-exposure comparison

To separate portfolio-construction effects from exposure effects, the independent sleeve targets were rescaled on each rebalance date to match the composite portfolio's target gross exposure. The exposure audit confirms an exact match, with average target gross exposure of approximately 1.82 for both portfolios.

At matched exposure, the independent sleeves produce a slightly higher gross annualised return (15.53% versus 15.19%) and gross Sharpe ratio (0.823 versus 0.797). The composite portfolio, however, has lower average rebalance turnover (0.477 versus 0.574) and lower transaction costs.

Consequently, their net performance is effectively identical: net Sharpe ratios are 0.678 for the independent sleeves and 0.679 for the composite. The composite also has a modestly smaller maximum drawdown (-30.63% versus -31.73%).

The two portfolios have a return correlation of approximately 0.95, while their average same-side target-weight overlap is approximately 65%. They therefore express broadly similar factor information but retain meaningful differences in stock selection and weighting.

Overall, neither construction method is decisively superior at matched exposure. The composite score is marginally more implementation-efficient, while the independent sleeves preserve clearer factor-level attribution. The original sleeve portfolio's smoother risk profile primarily results from natural netting and lower realised gross exposure.

## Dynamic risk-balanced sleeve portfolio

We build a dynamic independent-sleeve portfolio with:

$$
\hat{\sigma}_{k,t}
=
\operatorname{Std}\left(r_{k,t-63:t-1}\right),
\qquad
a_{k,t}
\propto
\frac{1}{\hat{\sigma}_{k,t}}.
$$


In [23]:
from alpha_research.portfolio import (
    estimate_trailing_sleeve_volatility,
    calculate_inverse_volatility_allocations,
    combine_dynamic_sleeve_target_weights,
)

In [24]:
def extract_active_gross_returns(
    daily: pd.DataFrame,
    exposure_tolerance: float = 1e-12,
) -> pd.Series:
    indexed = (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .set_index("date")
        .sort_index()
    )

    return indexed["gross_return"].where(
        indexed["gross_exposure"] > exposure_tolerance
    )


momentum_daily = audit_results[
    "12-1 Momentum"
]["drift_daily"]

volatility_daily = audit_results[
    "Realised Volatility"
]["drift_daily"]

sleeve_return_frame = pd.concat(
    {
        "Momentum": extract_active_gross_returns(
            momentum_daily
        ),
        "Realised Volatility": extract_active_gross_returns(
            volatility_daily
        ),
    },
    axis=1,
).sort_index()

trailing_sleeve_volatility = (
    estimate_trailing_sleeve_volatility(
        sleeve_returns=sleeve_return_frame,
        lookback=63,
        min_periods=42,
        periods_per_year=252,
    )
)

daily_risk_allocations = (
    calculate_inverse_volatility_allocations(
        sleeve_volatility=trailing_sleeve_volatility,
        allocation_floor=0.20,
    )
)

#### Align allocations with rebalance dates

In [25]:
rebalance_dates = pd.DatetimeIndex(
    combined_targets["date"].unique()
).sort_values()

dynamic_sleeve_allocations = (
    daily_risk_allocations
    .reindex(rebalance_dates, method="ffill")
)

if dynamic_sleeve_allocations.isna().any().any():
    raise ValueError(
        "Dynamic allocations are missing rebalance dates."
    )

dynamic_sleeve_allocations.index.name = "date"

dynamic_allocation_summary = (
    dynamic_sleeve_allocations
    .agg(["mean", "std", "min", "max"])
    .T
)

dynamic_allocation_summary[
    "mean_absolute_change"
] = (
    dynamic_sleeve_allocations
    .diff()
    .abs()
    .mean()
)

dynamic_allocation_summary.round(4)

,mean,std,min,max,mean_absolute_change
Momentum,0.5128,0.0474,0.3737,0.6457,0.0056
Realised Volatility,0.4872,0.0474,0.3543,0.6263,0.0056


In [26]:
rebalance_volatility = (
    trailing_sleeve_volatility
    .reindex(rebalance_dates, method="ffill")
)

valid_risk_estimate = (
    rebalance_volatility.notna().all(axis=1)
)

risk_allocation_start = (
    valid_risk_estimate[valid_risk_estimate]
    .index.min()
)

print("First risk-based allocation date:", risk_allocation_start)
print(
    "Risk-based rebalance fraction:",
    round(valid_risk_estimate.mean(), 4),
)

First risk-based allocation date: 2016-03-14 00:00:00
Risk-based rebalance fraction: 0.8962


#### Construct and backtest the dynamic portfolio

In [27]:
dynamic_sleeve_targets = (
    combine_dynamic_sleeve_target_weights(
        sleeve_targets={
            "Momentum": momentum_targets,
            "Realised Volatility": volatility_targets,
        },
        sleeve_allocations=dynamic_sleeve_allocations,
    )
)

risk_balanced_daily, risk_balanced_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=dynamic_sleeve_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )
)

#### Examine the simpler inverse-volatility risk proxy a_k * sig_k

In [28]:
allocated_volatility_proxy = (
    dynamic_sleeve_allocations
    * rebalance_volatility
)

proxy_risk_shares = allocated_volatility_proxy.div(
    allocated_volatility_proxy.sum(axis=1),
    axis=0,
)

proxy_risk_share_summary = (
    proxy_risk_shares
    .loc[valid_risk_estimate]
    .agg(["mean", "std", "min", "max"])
    .T
)

proxy_risk_share_summary.round(4)

,mean,std,min,max
Momentum,0.4899,0.0355,0.3868,0.5943
Realised Volatility,0.5101,0.0355,0.4057,0.6132


#### Calculate covariance-aware ex-ante risk contributions

In [29]:
risk_contribution_rows = []

for date in rebalance_dates:
    history = (
        sleeve_return_frame.loc[sleeve_return_frame.index < date].dropna().tail(63)
    )

    if len(history) < 42:
        continue

    covariance = history.cov() * 252

    weights = dynamic_sleeve_allocations.loc[date, covariance.columns]

    portfolio_variance = float(
        weights.to_numpy() @ covariance.to_numpy() @ weights.to_numpy()
    )

    if portfolio_variance <= 0.0:
        continue

    marginal_variance = covariance.to_numpy() @ weights.to_numpy()

    contribution_shares = weights.to_numpy() * marginal_variance / portfolio_variance

    row = {"date": date}

    for sleeve_name, contribution in zip(
        covariance.columns,
        contribution_shares,
    ):
        row[sleeve_name] = contribution

    risk_contribution_rows.append(row)

risk_contribution_shares = pd.DataFrame(risk_contribution_rows).set_index("date")

risk_contribution_summary = risk_contribution_shares.agg(
    ["mean", "std", "min", "max"]
).T

risk_contribution_summary.round(4)


,mean,std,min,max
Momentum,0.4538,0.1326,0.0246,0.8980
Realised Volatility,0.5462,0.1326,0.1020,0.9754


#### Compare portfolio performance

In [30]:
risk_balance_comparison_daily = {
    "Momentum": momentum_daily,
    "Realised Volatility": volatility_daily,
    "Fixed 50/50 Sleeves": combined_daily,
    "Dynamic Risk-Balanced Sleeves": risk_balanced_daily,
    "Composite Score": composite_daily,
}

risk_balance_summary_rows = []

for portfolio_name, daily in (
    risk_balance_comparison_daily.items()
):
    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    risk_balance_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross_summary[
                "annualised_return"
            ],
            "net_annualised_return": net_summary[
                "annualised_return"
            ],
            "net_annualised_volatility": net_summary[
                "annualised_volatility"
            ],
            "gross_sharpe": gross_summary[
                "sharpe_ratio"
            ],
            "net_sharpe": net_summary[
                "sharpe_ratio"
            ],
            "max_drawdown": net_summary[
                "max_drawdown"
            ],
            "average_rebalance_turnover": net_summary[
                "average_rebalance_turnover"
            ],
            "total_transaction_cost": net_summary[
                "total_transaction_cost"
            ],
            "average_daily_gross_exposure": daily[
                "gross_exposure"
            ].mean(),
        }
    )

risk_balance_summary = (
    pd.DataFrame(risk_balance_summary_rows)
    .set_index("portfolio")
)

risk_balance_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
Fixed 50/50 Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
Dynamic Risk-Balanced Sleeves,0.1227,0.0988,0.1574,0.8143,0.6777,-0.2103,0.4261,0.2463,1.5074
Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [31]:
risk_balance_gross_returns = pd.concat(
    {
        portfolio_name: daily.set_index("date")[
            "gross_return"
        ]
        for portfolio_name, daily
        in risk_balance_comparison_daily.items()
    },
    axis=1,
)

risk_balance_gross_returns.corr().round(4)

,Momentum,Realised Volatility,Fixed 50/50 Sleeves,Dynamic Risk-Balanced Sleeves,Composite Score
Momentum,1.0000,0.0225,0.6696,0.7154,0.5510
Realised Volatility,0.0225,1.0000,0.7572,0.7042,0.7886
Fixed 50/50 Sleeves,0.6696,0.7572,1.0000,0.9905,0.9457
Dynamic Risk-Balanced Sleeves,0.7154,0.7042,0.9905,1.0000,0.9325
Composite Score,0.5510,0.7886,0.9457,0.9325,1.0000


#### Inspect dynamic target exposure

In [ ]:
dynamic_target_exposure = (
    dynamic_sleeve_targets
    .assign(
        long_weight=lambda df: df["weight"].clip(
            lower=0.0
        ),
        short_weight=lambda df: -df["weight"].clip(
            upper=0.0
        ),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
    )
)

dynamic_target_exposure["gross_exposure"] = (
    dynamic_target_exposure["long_exposure"]
    + dynamic_target_exposure["short_exposure"]
)

dynamic_target_exposure["net_exposure"] = (
    dynamic_target_exposure["long_exposure"]
    - dynamic_target_exposure["short_exposure"]
)

dynamic_target_exposure.agg(
    ["mean", "std", "min", "max"]
).round(4)

,long_exposure,short_exposure,gross_exposure,net_exposure
mean,0.7533,0.7533,1.5065,-0.0
std,0.1871,0.1871,0.3741,0.0
min,0.0000,0.0000,0.0000,-0.0
max,1.0000,1.0000,2.0000,0.0


### Experiment 4: Dynamic risk-balanced factor sleeves

The dynamic sleeve portfolio replaces the fixed 50/50 allocation with bounded inverse-volatility weights estimated from trailing, one-day-shifted sleeve returns. Risk-based allocations begin on 14 March 2016 and cover approximately 89.6% of rebalance dates, with equal allocations used during the warm-up period.

The resulting allocations remain moderate and stable. Momentum receives an average allocation of 51.3% and realised volatility 48.7%, while the average absolute allocation change is only 0.56% per rebalance. The volatility-based risk proxy is close to balanced, although covariance-aware contributions average 45.4% for momentum and 54.6% for realised volatility. This difference reflects the fact that inverse-volatility allocation does not explicitly incorporate time-varying covariance.

Relative to the fixed 50/50 sleeves, dynamic allocation modestly improves net annualised return from 9.82% to 9.88% and reduces annualised volatility from 16.16% to 15.74%. Net Sharpe increases from 0.661 to 0.678, while maximum drawdown improves materially from -25.66% to -21.03%. These benefits come with only slightly higher turnover and transaction costs.

The fixed and dynamic portfolios have a return correlation of approximately 0.99, showing that risk balancing refines the existing factor combination rather than introducing a distinct return source. Its principal benefit is a smoother risk path and improved drawdown control, rather than a large increase in headline performance.

### Subperiod analysis

In [33]:
subperiod_definitions = {
    "2015-2018": (
        pd.Timestamp("2015-01-01"),
        pd.Timestamp("2018-12-31"),
    ),
    "2019-2022": (
        pd.Timestamp("2019-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "2023-Present": (
        pd.Timestamp("2023-01-01"),
        None,
    ),
}

subperiod_portfolios = {
    "Fixed 50/50 Sleeves": combined_daily,
    "Dynamic Risk-Balanced Sleeves": risk_balanced_daily,
    "Composite Score": composite_daily,
}

subperiod_portfolios = {
    name: (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .sort_values("date")
        .reset_index(drop=True)
    )
    for name, daily in subperiod_portfolios.items()
}

exposure_tolerance = 1e-12

active_start_dates = {
    name: daily.loc[
        daily["gross_exposure"] > exposure_tolerance,
        "date",
    ].min()
    for name, daily in subperiod_portfolios.items()
}

if any(pd.isna(date) for date in active_start_dates.values()):
    raise ValueError("At least one portfolio never becomes active.")

common_evaluation_start = max(active_start_dates.values())

print("Individual active starts:")
for name, date in active_start_dates.items():
    print(f"  {name}: {date.date()}")

print(
    "Common evaluation start:",
    common_evaluation_start.date(),
)

Individual active starts:
  Fixed 50/50 Sleeves: 2015-04-08
  Dynamic Risk-Balanced Sleeves: 2015-04-08
  Composite Score: 2016-01-07
Common evaluation start: 2016-01-07


In [ ]:
subperiod_summary_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    for portfolio_name, daily in subperiod_portfolios.items():
        mask = daily["date"] >= evaluation_start

        if period_end is not None:
            mask &= daily["date"] <= period_end

        period_daily = daily.loc[mask].copy()

        if period_daily.empty:
            continue

        gross_summary = summarise_backtest(
            period_daily,
            return_column="gross_return",
        ).iloc[0]

        net_summary = summarise_backtest(
            period_daily,
            return_column="net_return",
        ).iloc[0]

        subperiod_summary_rows.append(
            {
                "period": period_name,
                "portfolio": portfolio_name,
                "start_date": period_daily["date"].min(),
                "end_date": period_daily["date"].max(),
                "observations": len(period_daily),
                "gross_annualised_return": gross_summary["annualised_return"],
                "net_annualised_return": net_summary["annualised_return"],
                "net_annualised_volatility": net_summary["annualised_volatility"],
                "gross_sharpe": gross_summary["sharpe_ratio"],
                "net_sharpe": net_summary["sharpe_ratio"],
                "max_drawdown": net_summary["max_drawdown"],
                "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
                "total_transaction_cost": net_summary["total_transaction_cost"],
                "average_daily_gross_exposure": period_daily["gross_exposure"].mean(),
            }
        )

subperiod_summary = pd.DataFrame(subperiod_summary_rows).set_index(
    ["period", "portfolio"]
)

subperiod_summary.round(4)

C:\Users\39521\AppData\Local\Temp\ipykernel_52956\3449113191.py:76: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  subperiod_summary.round(4)


start_date   end_date  \
period       portfolio                                             
2015-2018    Fixed 50/50 Sleeves           2016-01-07 2018-12-31   
             Dynamic Risk-Balanced Sleeves 2016-01-07 2018-12-31   
             Composite Score               2016-01-07 2018-12-31   
2019-2022    Fixed 50/50 Sleeves           2019-01-02 2022-12-30   
             Dynamic Risk-Balanced Sleeves 2019-01-02 2022-12-30   
             Composite Score               2019-01-02 2022-12-30   
2023-Present Fixed 50/50 Sleeves           2023-01-03 2026-07-01   
             Dynamic Risk-Balanced Sleeves 2023-01-03 2026-07-01   
             Composite Score               2023-01-03 2026-07-01   

                                            observations  \
period       portfolio                                     
2015-2018    Fixed 50/50 Sleeves                     751   
             Dynamic Risk-Balanced Sleeves           751   
             Composite Score                         751   
2019-2022    Fixed 50/50 Sleeves                    1008   
             Dynamic Risk-Balanced Sleeves          1008   
             Composite Score                        1008   
2023-Present Fixed 50/50 Sleeves                     876   
             Dynamic Risk-Balanced Sleeves           876   
             Composite Score                         876   

                                            gross_annualised_return  \
period       portfolio                                                
2015-2018    Fixed 50/50 Sleeves                             0.0557   
             Dynamic Risk-Balanced Sleeves                   0.0519   
             Composite Score                                 0.0735   
2019-2022    Fixed 50/50 Sleeves                             0.0345   
             Dynamic Risk-Balanced Sleeves                   0.0533   
             Composite Score                                 0.0431   
2023-Present Fixed 50/50 Sleeves                             0.3190   
             Dynamic Risk-Balanced Sleeves                   0.3011   
             Composite Score                                 0.4293   

                                            net_annualised_return  \
period       portfolio                                              
2015-2018    Fixed 50/50 Sleeves                           0.0323   
             Dynamic Risk-Balanced Sleeves                 0.0281   
             Composite Score                               0.0460   
2019-2022    Fixed 50/50 Sleeves                           0.0126   
             Dynamic Risk-Balanced Sleeves                 0.0299   
             Composite Score                               0.0151   
2023-Present Fixed 50/50 Sleeves                           0.2901   
             Dynamic Risk-Balanced Sleeves                 0.2718   
             Composite Score                               0.3928   

                                            net_annualised_volatility  \
period       portfolio                                                  
2015-2018    Fixed 50/50 Sleeves                               0.1369   
             Dynamic Risk-Balanced Sleeves                     0.1362   
             Composite Score                                   0.1684   
2019-2022    Fixed 50/50 Sleeves                               0.1538   
             Dynamic Risk-Balanced Sleeves                     0.1438   
             Composite Score                                   0.2019   
2023-Present Fixed 50/50 Sleeves                               0.2048   
             Dynamic Risk-Balanced Sleeves                     0.2028   
             Composite Score                                   0.2560   

                                            gross_sharpe  net_sharpe  \
period       portfolio                                                 
2015-2018    Fixed 50/50 Sleeves                  0.4649      0.3010   
             Dynamic Risk-Balanced Sleeves        0.4400      0.2721   

#### Calculate the dynamic portfolio’s relative performance

In [ ]:
subperiod_delta_metrics = [
    "net_annualised_return",
    "net_annualised_volatility",
    "net_sharpe",
    "max_drawdown",
    "average_rebalance_turnover",
    "total_transaction_cost",
    "average_daily_gross_exposure",
]

dynamic_subperiod_summary = subperiod_summary.xs(
    "Dynamic Risk-Balanced Sleeves",
    level="portfolio",
)[subperiod_delta_metrics]

fixed_subperiod_summary = subperiod_summary.xs(
    "Fixed 50/50 Sleeves",
    level="portfolio",
)[subperiod_delta_metrics]

composite_subperiod_summary = subperiod_summary.xs(
    "Composite Score",
    level="portfolio",
)[subperiod_delta_metrics]

dynamic_minus_fixed = dynamic_subperiod_summary - fixed_subperiod_summary

dynamic_minus_composite = dynamic_subperiod_summary - composite_subperiod_summary

print("Dynamic minus Fixed 50/50:")
display(dynamic_minus_fixed.round(4))

print("Dynamic minus Composite:")
display(dynamic_minus_composite.round(4))

Dynamic minus Fixed 50/50:


,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
period,,,,,,,
2015-2018,-0.0042,-0.0008,-0.0288,-0.0003,0.0078,0.0012,0.0235
2019-2022,0.0173,-0.0099,0.1183,0.0540,0.0212,0.0043,0.0634
2023-Present,-0.0183,-0.0019,-0.0595,-0.0002,0.0121,0.0021,0.0261


Dynamic minus Composite:


,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
period,,,,,,,
2015-2018,-0.0179,-0.0322,-0.0797,0.0715,-0.0612,-0.0092,-0.3677
2019-2022,0.0147,-0.0580,0.1012,0.1037,-0.0929,-0.0187,-0.4956
2023-Present,-0.1210,-0.0531,-0.1362,0.0555,-0.0615,-0.0108,-0.3764


#### Summarise dynamic allocations by subperiod

In [ ]:
allocation_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    allocation_mask = dynamic_sleeve_allocations.index >= evaluation_start

    if period_end is not None:
        allocation_mask &= dynamic_sleeve_allocations.index <= period_end

    period_allocations = dynamic_sleeve_allocations.loc[allocation_mask]

    if period_allocations.empty:
        continue

    for sleeve_name in period_allocations.columns:
        sleeve_allocation = period_allocations[sleeve_name]

        allocation_subperiod_rows.append(
            {
                "period": period_name,
                "sleeve": sleeve_name,
                "mean_allocation": sleeve_allocation.mean(),
                "allocation_std": sleeve_allocation.std(),
                "minimum_allocation": sleeve_allocation.min(),
                "maximum_allocation": sleeve_allocation.max(),
                "mean_absolute_change": (sleeve_allocation.diff().abs().mean()),
            }
        )

allocation_subperiod_summary = pd.DataFrame(allocation_subperiod_rows).set_index(
    ["period", "sleeve"]
)

allocation_subperiod_summary.round(4)

mean_allocation  allocation_std  \
period       sleeve                                                 
2015-2018    Momentum                      0.4935          0.0333   
             Realised Volatility           0.5065          0.0333   
2019-2022    Momentum                      0.5280          0.0617   
             Realised Volatility           0.4720          0.0617   
2023-Present Momentum                      0.5157          0.0382   
             Realised Volatility           0.4843          0.0382   

                                  minimum_allocation  maximum_allocation  \
period       sleeve                                                        
2015-2018    Momentum                         0.4373              0.5769   
             Realised Volatility              0.4231              0.5627   
2019-2022    Momentum                         0.3737              0.6457   
             Realised Volatility              0.3543              0.6263   
2023-Present Momentum                         0.4569              0.6059   
             Realised Volatility              0.3941              0.5431   

                                  mean_absolute_change  
period       sleeve                                     
2015-2018    Momentum                           0.0058  
             Realised Volatility                0.0058  
2019-2022    Momentum                           0.0068  
             Realised Volatility                0.0068  
2023-Present Momentum                           0.0057  
             Realised Volatility                0.0057

In [ ]:
risk_estimate_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    period_mask = valid_risk_estimate.index >= evaluation_start

    if period_end is not None:
        period_mask &= valid_risk_estimate.index <= period_end

    period_validity = valid_risk_estimate.loc[period_mask]

    if period_validity.empty:
        continue

    risk_estimate_subperiod_rows.append(
        {
            "period": period_name,
            "rebalance_dates": len(period_validity),
            "risk_based_rebalance_fraction": (period_validity.mean()),
        }
    )

risk_estimate_subperiod_summary = pd.DataFrame(risk_estimate_subperiod_rows).set_index(
    "period"
)

risk_estimate_subperiod_summary.round(4)

,rebalance_dates,risk_based_rebalance_fraction
period,,
2015-2018,151,0.9404
2019-2022,201,1.0000
2023-Present,175,1.0000


#### Examine covariance-aware risk contributions

In [38]:
risk_contribution_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    contribution_mask = risk_contribution_shares.index >= evaluation_start

    if period_end is not None:
        contribution_mask &= risk_contribution_shares.index <= period_end

    period_contributions = risk_contribution_shares.loc[contribution_mask]

    if period_contributions.empty:
        continue

    for sleeve_name in period_contributions.columns:
        sleeve_contribution = period_contributions[sleeve_name]

        risk_contribution_subperiod_rows.append(
            {
                "period": period_name,
                "sleeve": sleeve_name,
                "mean_risk_contribution": (sleeve_contribution.mean()),
                "risk_contribution_std": (sleeve_contribution.std()),
                "minimum_risk_contribution": (sleeve_contribution.min()),
                "maximum_risk_contribution": (sleeve_contribution.max()),
            }
        )

risk_contribution_subperiod_summary = pd.DataFrame(
    risk_contribution_subperiod_rows
).set_index(["period", "sleeve"])

risk_contribution_subperiod_summary.round(4)


mean_risk_contribution  \
period       sleeve                                        
2015-2018    Momentum                             0.4799   
             Realised Volatility                  0.5201   
2019-2022    Momentum                             0.4117   
             Realised Volatility                  0.5883   
2023-Present Momentum                             0.4809   
             Realised Volatility                  0.5191   

                                  risk_contribution_std  \
period       sleeve                                       
2015-2018    Momentum                            0.0890   
             Realised Volatility                 0.0890   
2019-2022    Momentum                            0.1738   
             Realised Volatility                 0.1738   
2023-Present Momentum                            0.0882   
             Realised Volatility                 0.0882   

                                  minimum_risk_contribution  \
period       sleeve                                           
2015-2018    Momentum                                0.1385   
             Realised Volatility                     0.4434   
2019-2022    Momentum                                0.0246   
             Realised Volatility                     0.2517   
2023-Present Momentum                                0.2373   
             Realised Volatility                     0.1020   

                                  maximum_risk_contribution  
period       sleeve                                          
2015-2018    Momentum                                0.5566  
             Realised Volatility                     0.8615  
2019-2022    Momentum                                0.7483  
             Realised Volatility                     0.9754  
2023-Present Momentum                                0.8980  
             Realised Volatility                     0.7627

### Experiment 4.1: Subperiod robustness

The three main portfolio constructions were compared over 2016–2018, 2019–2022, and 2023–present using a common evaluation start of 7 January 2016. The first period therefore excludes 2015 because the composite signal was not yet active.

Dynamic risk balancing does not consistently outperform the fixed 50/50 allocation. Its clearest benefit occurs during 2019–2022, when it improves net annualised return by 1.73 percentage points, reduces volatility by 0.99 percentage points, raises net Sharpe by 0.118, and reduces maximum drawdown from -25.66% to -20.26%. In 2016–2018 and 2023–present, however, the fixed allocation produces slightly higher return and Sharpe, with almost identical drawdowns.

Consequently, the dynamic portfolio's modest full-sample improvement is largely attributable to the 2019–2022 period rather than a persistent advantage across regimes. It should therefore be interpreted as a defensive risk-management refinement, not a reliable factor-timing mechanism.

Relative to the composite score, the dynamic sleeves consistently carry lower volatility, gross exposure, turnover, transaction costs, and drawdown. The composite nevertheless produces higher return and Sharpe in 2016–2018 and 2023–present. Part of this return difference reflects its materially higher gross exposure, averaging approximately 2.00 versus 1.51–1.63 for the dynamic portfolio.

Dynamic allocations remain moderate across all periods, but covariance-aware risk contributions are not consistently balanced. The imbalance is greatest during 2019–2022, despite the method's strongest performance improvement. The current approach therefore balances estimated standalone sleeve volatility more closely than total portfolio risk.

Overall, fixed 50/50 sleeves remain the simplest and most robust baseline. Dynamic risk balancing is retained as a useful alternative with better full-sample downside behaviour, while the composite remains the more aggressive, implementation-efficient construction.